# UFLD ensemble: conformal calibration and inference

Inference-only ensemble: one pretrained Transformer and one PPO agent per seed (trained with `0-pretrain.ipynb` and `1-rl.ipynb`). The members' sigmoid outputs are averaged at every step, and the conformal stopping rule is applied to the averaged probabilities.

In [ ]:
import os
import sys

# Run from the repository root so that `src/`, `data/` and `models/` resolve correctly
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

In [ ]:
import pickle

import torch

from src.config import MODEL_CONFIGS, classifier_checkpoint, ppo_model_name
from src.models.classifier.resnet import init_resnet
from src.models.patch_selection.classifier import CausalTransformerClassifierKVCached
from src.models.patch_selection.environment_ensemble import WaferMapGridEnsemble
from src.models.patch_selection.ppo_ensemble import PPOAgentEnsemble
from src.utils.conformal import (
    make_AB_loaders,
    calibrate_multilabel_CP_two_rule_AB,
    evaluate_multilabel_CP_two_rule,
)
from src.utils.dataloader import load_mixedwm38_splits, get_loaders
from src.utils.utils import seed_everything

## Configuration

In [ ]:
SEEDS = [0, 1, 2, 3, 4]   # one ensemble member per seed
HYBRID = False            # fall back to ResNet-34 when the stopping rule never fires
CNN_SEED = 2              # seed of the ResNet-34 used in the hybrid setting
MODEL_SIZE = 'small'      # 'small' | 'medium' | 'large' (Table 1)
ALPHA = 0.065             # miscoverage level

DATA_PATH = 'data/MixedWM38.npz'
RESULTS_DIR = 'results'

# Stopping-rule settings shared by calibration (set B) and evaluation
CP_KWARGS = dict(m=4, min_steps=1, patience=1, max_steps=None,
                 break_ties_during_run=False, ensemble=True)

seed_everything(seed=2024)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
os.makedirs(RESULTS_DIR, exist_ok=True)

dataset_name = 'mixedwm38'
num_classes = 8
input_channels = 1
img_size = (52, 52)
batch_size = 128

## Data

In [ ]:
train_dataset, val_dataset, calib_dataset, test_dataset = load_mixedwm38_splits(
    DATA_PATH, img_size=img_size, input_channels=input_channels, num_classes=num_classes)

train_loader, test_loader, val_loader, calib_loader = get_loaders(
    train_dataset, test_dataset, val_dataset=val_dataset, calib_dataset=calib_dataset, batch_size=batch_size)

## Ensemble members

In [ ]:
embedding_dim, num_layers = MODEL_CONFIGS[MODEL_SIZE]
hidden_dim = 2 * embedding_dim
num_heads = 4
patch_size = 4
num_patches = (img_size[0] // patch_size) ** 2   # 169

classifiers = []
for seed in SEEDS:
    classifier = CausalTransformerClassifierKVCached(
        input_dim=patch_size ** 2,
        hidden_dim=hidden_dim,
        num_classes=num_classes,
        seq_len=num_patches,
        num_layers=num_layers,
        num_heads=num_heads,
        patch_size=patch_size,
        dropout_prob=0.1,
        batch_first=True,
        layer_norm=True,
        use_cls_token=False,
        embedding_dim=embedding_dim,
        use_patch_id_emb=True,
    )
    classifier.load_weights(classifier_checkpoint(dataset_name, seed, num_layers, hidden_dim, num_heads=num_heads))
    classifier = classifier.to(device)
    classifier.eval()
    classifier.freeze_all()
    classifiers.append(classifier)

In [ ]:
env = WaferMapGridEnsemble(classifiers, val_loader, test_loader, start_policy=0, reward_fun="sparse", device=device)

# lr, gamma, gae_lambda and the seeds are part of the checkpoint file names
lr, gamma, gae_lambda = 1e-3, 0.99, 0.99
agent = PPOAgentEnsemble(
    n_actions=env.action_space.n, obs_dim=classifiers[0].embed_dim,
    batch_size=batch_size, gamma=gamma, alpha=lr, gae_lambda=gae_lambda,
    n_epochs=100, policy_clip=0.2, seeds=SEEDS, chkpt_dir='models',
    model_name=ppo_model_name(dataset_name, num_layers, hidden_dim, num_heads=num_heads),
                 device=device)
agent.load_models()

In [ ]:
if HYBRID:
    cnn_model = init_resnet('resnet34', num_classes=num_classes, input_channels=input_channels, pretrained=False)
    cnn_model.load_state_dict(torch.load(f"models/{dataset_name}/resnet34_seed{CNN_SEED}.pth"))
    cnn_model = cnn_model.to(device)
    cnn_model.eval()
else:
    cnn_model = None

## Conformal calibration (A/B) and test evaluation

In [ ]:
calib_loader_A, calib_loader_B = make_AB_loaders(
    calib_dataset, batch_size=batch_size, split=0.5, shuffle=True, num_workers=4, pin_memory=True)

q0_A, q1_A, q0_B, q1_B = calibrate_multilabel_CP_two_rule_AB(
    agent=agent, env=env,
    calib_loader_A=calib_loader_A, calib_loader_B=calib_loader_B,
    alpha=ALPHA, device=device, **CP_KWARGS,
)

In [ ]:
test_results = evaluate_multilabel_CP_two_rule(
    agent=agent, env=env, test_loader=test_loader,
    q0=q0_B, q1=q1_B, cnn_model=cnn_model, device=device, **CP_KWARGS,
)

with open(os.path.join(RESULTS_DIR, f'ensemble-{MODEL_SIZE}_alpha{ALPHA}.pickle'), 'wb') as handle:
    pickle.dump(test_results, handle, protocol=pickle.HIGHEST_PROTOCOL)